## 0. 불러오기 & 컬럼 선택

In [1]:
import html
import pandas as pd
from pathlib import Path


def clean_text(s):
    """&amp;cr; 같은 기호 정리"""
    s = s.map(lambda x: html.unescape(x) if isinstance(x, str) else x)
    s = s.str.replace("&cr;", " ", regex=False)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    return s


df = pd.read_csv(Path("../data/merged/종속기업_원본.csv"), dtype=str)
df = df[["crno", "sbrdEnpNm", "sbrdEnpMainBizCtt", "sbrdEnpadr"]]
print(f"{len(df):,}행")

30,224행


## 1. 기호 정리
- 이름, 주요사업, 주소 컬럼에 있는 기호들을 정리한다.

In [2]:
for c in ["sbrdEnpNm", "sbrdEnpMainBizCtt", "sbrdEnpadr"]:
    df[c] = clean_text(df[c])

df.head()

,crno,sbrdEnpNm,sbrdEnpMainBizCtt,sbrdEnpadr
0,1101110015639,GYOZA KEIKAKU CO. LTD.,식료품 제조 및 판매업,105-0003 Tokyo Minato City Nishishinbashi 2 Ch...
1,1101110015639,CJ FOODS ASIA HOLDINGS LIMITED,지주업,Suite 3003 30/F Central Plaza 18 Harbour Road ...
2,1101110015639,CJ FOODS USA INC.,지주업,4 Centerpointe Dr. Suite 100 La Palma CA 90623
3,1101110015639,LOC TAN INVESTMENT COMPANY LIMITED,식료품 제조 및 판매업,VILLAGE 4 LOC TAN WARD ho chi minh vietnam
4,1101110015639,PT CHEIL JEDANG BIO INDONESIA,식품 및 사료첨가제 도매 유통업,Menara BP Jamsostek 21 floor Jl. Jend. Gatot S...


## 1-1. 상동 채우기
- 주소 컬럼에 있는 '상동' 항목들을 위에 있는 회사로 주소를 채운다.
    - `groupby`로 법인번호를 묶었기 때문에 같은 모회사 안에서만 채우게 했다.

In [3]:
adr = df["sbrdEnpadr"].fillna("").str.strip()
is_same = adr == "상동"
bad = adr.isin(["상동", "-", ""]) | adr.str.match(r"^\(?주\d*\)?$")

filled = df["sbrdEnpadr"].where(~bad).groupby(df["crno"]).ffill()
df.loc[is_same, "sbrdEnpadr"] = filled[is_same]

print("남은 상동:", (df["sbrdEnpadr"] == "상동").sum())

남은 상동: 0


## 1-2. 확인할 수 없음 채우기
- 주소 컬럼에서 주소가 없는 항목을 `확인할수없음`으로 채운다.

In [4]:
adr = df["sbrdEnpadr"].fillna("").str.strip()
no_adr = adr.isin(["-", "", "케이 글로우엔터테크 신기술사업투자조합"]) | adr.str.match(r"^\(?주\d*\)?$")

df.loc[no_adr, "sbrdEnpadr"] = "확인할수없음"
print("확인할수없음:", no_adr.sum())

확인할수없음: 4371


## 2. 이름 없는 행 & 중복 삭제

In [5]:
before = len(df)

# 이름이 없거나 "-"인 행 삭제
df = df[~df["sbrdEnpNm"].fillna("").isin(["", "-"])]
print(f"이름 없는 행 삭제 후: {len(df):,}행")

# 완전히 같은 행 삭제
df = df.drop_duplicates()
print(f"중복 삭제 후: {len(df):,}행")

print(f"총 {before - len(df):,}행 삭제")

이름 없는 행 삭제 후: 27,698행
중복 삭제 후: 27,697행
총 2,527행 삭제


## 3. 국내/해외/모름 표시

In [6]:
import re

KOREA_CITIES = (
    "수원|성남|의정부|안양|부천|광명|평택|동두천|안산|고양|과천|구리|남양주|오산|시흥|군포|의왕|하남|용인|파주|이천|안성|김포|화성|양주|포천|여주|가평|양평|연천"
    "|춘천|원주|강릉|동해|태백|속초|삼척|홍천|횡성|영월|평창|정선|철원|화천|양구|인제|고성|양양"
    "|청주|충주|제천|보은|옥천|영동|증평|진천|괴산|음성|단양"
    "|천안|공주|보령|아산|서산|논산|계룡|당진|금산|부여|서천|청양|홍성|예산|태안"
    "|전주|군산|익산|정읍|남원|김제|완주|진안|무주|장수|임실|순창|고창|부안"
    "|목포|여수|순천|나주|광양|담양|곡성|구례|고흥|보성|화순|장흥|강진|해남|영암|무안|함평|영광|장성|완도|진도|신안"
    "|포항|경주|김천|안동|구미|영주|영천|상주|문경|경산|군위|의성|청송|영양|영덕|청도|고령|성주|칠곡|예천|봉화|울진|울릉"
    "|창원|진주|통영|사천|김해|밀양|거제|양산|의령|함안|창녕|남해|하동|산청|함양|거창|합천"
    "|서귀포|울주|기장|달성|강화|옹진|강남|오송|기흥"
)

KOREA = re.compile(
    r"대한민국|대한 민국|한국|국내|서울|부산|대구|인천|광주|대전|울산|세종|경기|강원|충청|충북|충남|전라|전북|전남|경상|경북|경남|제주"
    r"|" + KOREA_CITIES +
    r"|[가-힣]+구 [가-힣0-9]+(로|길)"
    r"|Seoul|Busan|Daegu|Incheon|Gwangju|Daejeon|Ulsan|Sejong|Gyeonggi|Gangwon|Chungcheong|Jeolla|Gyeongsang|Jeju|Korea",
    re.I,
)

FOREIGN = re.compile(
    r"미국|중국|일본|베트남|인도네시아|태국|싱가포르|싱가폴|홍콩|대만|인도|필리핀|말레이시아|독일|영국|프랑스|네덜란드"
    r"|멕시코|브라질|캐나다|호주|러시아|폴란드|헝가리|체코|슬로바키아|터키|튀르키예|우즈베키스탄|카자흐스탄|미얀마"
    r"|캄보디아|방글라데시|스페인|이탈리아|루마니아|포르투갈|덴마크|칠레|스웨덴|스위스|두바이|아랍에미"
    r"|과테말라|콜롬비아|우크라이나|벨기에|파나마|아르헨티나|사우디|이집트|니카라과|케이만|개성"
    r"|도쿄|대련|연길|산동성|박닌성"
    r"|USA|United States|America|China|Japan|Tokyo|Vietnam|Viet Nam|Indonesia|Thailand|Singapore|Hong ?Kong"
    r"|Taiwan|India|Philippines|Malaysia|Germany|United Kingdom|France|Netherlands|Mexico|Brazil|Canada"
    r"|Australia|Russia|Poland|Hungary|Czech|Slovakia|Turkey|Uzbekistan|Kazakhstan|Myanmar|Cambodia|Spain|Italy|UAE|Dubai",
    re.I,
)


def domestic_flag(addr):
    if not isinstance(addr, str) or addr in ("", "-", "확인할수없음"):
        return "모름"
    if not re.search(r"[가-힣]", addr):    # 영문 주소 → 해외
        return "해외"
    if FOREIGN.search(addr):              # 한글인데 외국 이름 → 해외
        return "해외"
    if KOREA.search(addr):                # 한글인데 한국 지역 → 국내
        return "국내"
    return "해외"                          # 한글 주소인데 한국 지명 없음 → 해외
    

df["domestic"] = df["sbrdEnpadr"].map(domestic_flag)
df["domestic"].value_counts()

domestic
해외    13970
국내    11682
모름     2045
Name: count, dtype: int64

## 4. 모름 확인

In [7]:
check = df[
    (df["domestic"] == "해외")
    & df["sbrdEnpadr"].str.contains("[가-힣]", na=False)
    & ~df["sbrdEnpadr"].str.contains(FOREIGN, na=False)
    & (df["sbrdEnpadr"] != "확인할수없음")
]
print("규칙 없이 해외로 간 한글 주소:", len(check))
check["sbrdEnpadr"].value_counts().head(30)

pd.set_option("display.max_rows", 100)
check["sbrdEnpadr"].value_counts()

규칙 없이 해외로 간 한글 주소: 84


sbrdEnpadr
몽골 올란 바타르 시 수흐바타르 구 1동 아 워터 잠치 딘 고담즈 34 제이 타워 7층                                                            2
나이지리아 라고스                                                                                                   2
파키스탄                                                                                                        2
680021 하바롭스크 주 하바롭스크 시 레닌그라드스카야 거리 53번지 1동 303호 사무실                                                         2
몽골                                                                                                          2
오사카부 오사카시 중앙구 카와라마지 1-7-2                                                                                   1
바누아투 포트빌라                                                                                                   1
튀니지                                                                                                         1
울란바타르시 항올두렉 15동 스타디온 어르길 (17011) 이흐 몽골국거리 71 이마트 3층                                                         1

## 5. 해외 삭제 + 주소 삭제
규칙 없이 해외로 간 한글 주소: 84 -> **삭제**

In [8]:
before = len(df)

df = df[df["domestic"] != "해외"]
print(f"해외 삭제: {before - len(df):,}행 → 남은 {len(df):,}행")

df = df.drop(columns=["sbrdEnpadr"])
df["domestic"].value_counts()

해외 삭제: 13,970행 → 남은 13,727행


domestic
국내    11682
모름     2045
Name: count, dtype: int64

## 5-1. 같은 모회사 안 이름 중복 확인

In [9]:
import re

def norm_name(x):
    """비교용 이름: (주)/주식회사/㈜/띄어쓰기 제거 + 대문자"""
    x = re.sub(r"\(주\)|주식회사|㈜|\(유\)|유한회사", "", x)
    return re.sub(r"\s", "", x).upper()

df["name_norm"] = df["sbrdEnpNm"].map(norm_name)

dup = df[df.duplicated(["crno", "name_norm"], keep=False)]
print("이름 중복 행:", len(dup))
dup.sort_values(["crno", "name_norm"])[["crno", "sbrdEnpNm", "sbrdEnpMainBizCtt"]].head(20)

이름 중복 행: 0


,crno,sbrdEnpNm,sbrdEnpMainBizCtt


## 6. 저장

In [10]:
CLEAN_DIR = Path("../data/clean")
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(CLEAN_DIR / "종속기업_정리.csv", index=False, encoding="utf-8-sig")
print(f"저장 완료: {len(df):,}행")
df.head()

저장 완료: 13,727행


,crno,sbrdEnpNm,sbrdEnpMainBizCtt,domestic,name_norm
216,1101110017867,한솔페이퍼텍(주),펄프 종이 및 판지제조업,국내,한솔페이퍼텍
217,1101110017867,한솔피엔에스(주),지류유통 및 IT서비스,국내,한솔피엔에스
218,1101110017867,한솔코에버(주),산업자동화,국내,한솔코에버
219,1101110017867,한솔로지스틱스(주),종합물류 서비스,국내,한솔로지스틱스
220,1101110017867,한솔티씨에스(주),종합물류 서비스,국내,한솔티씨에스


### 상동 확인용

In [8]:
raw = pd.read_csv(Path("../data/merged/종속기업_원본.csv"), dtype=str)
adr = raw["sbrdEnpadr"].fillna("").str.strip()
idx = raw.index[adr == "상동"]

prev_adr = adr.shift(1)
prev_crno = raw["crno"].shift(1)

print("상동 행:", len(idx))
print("위 행이 다른 모회사:", (raw.loc[idx, "crno"] != prev_crno[idx]).sum())
print("위 행도 상동/각주/-:", prev_adr[idx].str.match(r"^(상동|-|\(?주\d*\)?)$").sum())

상동 행: 103
위 행이 다른 모회사: 0
위 행도 상동/각주/-: 101


In [9]:
bad = adr.isin(["상동", "-", ""]) | adr.str.match(r"^\(?주\d*\)?$")

# 진짜 주소만 남기고, 같은 모회사 안에서 아래로 채우기
filled = adr.where(~bad).groupby(raw["crno"]).ffill()

print("채워지는 상동:", filled[idx].notna().sum())
print("못 채우는 상동:", filled[idx].isna().sum())

채워지는 상동: 103
못 채우는 상동: 0
